## Criação de ambientes

In [ ]:
storageAccountName = "sugarecommerce-datalake"
sasToken = "sv=2024-11-04&ss=bfqt&srt=sco&sp=rwdlacupyx&se=2025-06-24T10:50:16Z&st=2025-05-30T23:43:16Z&spr=https&sig=p5gYouJwHRbVfQmkyuc8PbDIgeUvk0Gce0nDKiwt6eA%3D"

def mount_adls(blobContainerName):
    try:
        dbutils.fs.mount(
            source=f"wasbs://{blobContainerName}@{storageAccountName}.blob.core.windows.net",
            mount_point=f"/mnt/{storageAccountName}/{blobContainerName}",
            extra_configs={f"fs.azure.sas.{blobContainerName}.{storageAccountName}.blob.core.windows.net": sasToken}
        )
        print(f"Montagem realizada: {blobContainerName}")
    except Exception as e:
        print(f"Já montado ou erro: {blobContainerName} -> {str(e)}")

In [ ]:
mount_adls('landing')
mount_adls('bronze')
mount_adls('silver')
mount_adls('gold')

In [ ]:
display(dbutils.fs.mounts())

In [ ]:
display(dbutils.fs.ls(f"/mnt/{storageAccountName}/landing-zone"))

## Salvando arquivos CSV das tabelas em formato Delta

In [ ]:
from pyspark.sql.functions import current_timestamp, lit

storageAccountName = "datalakef085704a8c687020"
landing_base = f"/mnt/{storageAccountName}/landing-zone"
bronze_base = f"/mnt/{storageAccountName}/bronze"

# Tabelas do modelo relacional OLTP
tabelas = {
    "customer": "customer.csv",
    "address": "address.csv",
    "category": "category.csv",
    "supplier": "supplier.csv",
    "product": "product.csv",
    "order": "order.csv",
    "order_item": "order_item.csv",
    "payment": "payment.csv",
    "promotion": "promotion.csv",
    "review": "review.csv",
    "product_promotion": "product_promotion.csv"
}

for nome_tabela, nome_arquivo in tabelas.items():
    try:
        print(f"📥 Lendo '{nome_arquivo}' para a tabela '{nome_tabela}'")

        df = (
            spark.read
            .option("inferSchema", True)
            .option("header", True)
            .csv(f"{landing_base}/{nome_arquivo}")
            .withColumn("data_hora_bronze", current_timestamp())
            .withColumn("nome_arquivo", lit(nome_arquivo))
        )

        caminho_destino = f"{bronze_base}/{nome_tabela}"
        df.write.format("delta").mode("overwrite").save(caminho_destino)

        print(f"Tabela '{nome_tabela}' salva em: {caminho_destino}")
    except Exception as e:
        print(f"Erro na tabela '{nome_tabela}': {str(e)}")


In [ ]:
display(dbutils.fs.ls(f"/mnt/{storageAccountName}/bronze/"))

In [ ]:
spark.read.format('delta').load(f'/mnt/{storageAccountName}/bronze/endereco').limit(10).display()